# Bank Earnings-Call NLP Pipeline — Loading, Structuring & Exploratory Analysis

We can separate the project into stages.

**Stage 1 — Collect**
Find suitable quarterly results material for one or two G-SIBs (Global Systemically Important Banks).

**Stage 2 — Structure**
Turn documents or transcripts into a consistent dataset.

**Stage 3 — Understand**
Explore coverage, speakers, sections, text lengths and data quality.

**Stage 4 — Extract signals** *(later notebook)*
Apply methods such as topic identification, financial sentiment analysis, keyword / metric extraction, summarisation.

**Stage 5 — Compare** *(later notebook)*
Compare topics within a bank, quarters within a bank, analysts versus management, one bank against a peer.

**Stage 6 — Interpret** *(later notebook)*
Ask whether the extracted information could genuinely provide additional insight into the firm's risk profile. A technically interesting NLP model is not automatically a useful regulatory insight.

---

**This notebook covers the hand-off from Stage 1 through Stage 2 and Stage 3.** Stage 1 (collection of the JPM and UBS transcripts into `all_sentences.csv`, `all_utterances.csv` and `all_metrics.csv`) has already happened upstream of this notebook. What follows here is:

1. Load the raw extracted CSVs and do a first sanity check that the upload actually worked.
2. Run exploratory data analysis on the raw data.
3. Build a small, reusable **Stage 2 structuring pipeline** (a handful of pure functions) that turns the raw tables into a clean, well-keyed dataset.
4. Run a confirmatory **Stage 3 pass** on the *structured* output, so anyone else on the team picking this up can trust it before it is used for topic modelling or sentiment analysis in Stage 4.

The structured CSVs this notebook produces (`structured_sentences.csv`, `structured_utterances.csv`, `qa_exchanges_utterance_level.csv`) are the hand-off artefact for the rest of the team.

# Setup

In [ ]:
import sys
import os
import pathlib
import pandas as pd
import re

# Visual libraries
import matplotlib.pyplot as plt
import seaborn as sns



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#----------------------------
# Reloads automatically
#----------------------------

%load_ext autoreload
%autoreload 2


In [ ]:
#-----------------------------------------------------------------------------------
# Properly add the project root to the path so python can find 'src'
#-----------------------------------------------------------------------------------


def find_project_root(marker: str = "src") -> pathlib.Path:
    path = pathlib.Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).is_dir():
            return parent
    raise RuntimeError(f"Could not find project root (looking for '{marker}') above {path}")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [14]:
#-----------------------------------------------------------------------------------
# Point DATA strictly to your local "data" directory
#-----------------------------------------------------------------------------------

DATA = project_root / "data"

## Load the raw data

We load the three raw tables produced by Stage 1: sentence-level, utterance-level (one row per
speaker turn) and metrics (reported financial figures per call). All three are keyed on a
particular export snapshot date, `EXPORT`, so that re-running this notebook later against a newer
extraction does not silently change the numbers under us.

In [35]:
from src import loading


In [16]:
EXPORT = "2026-09-13"

raw_utterances = loading.load(DATA, "all_utterances.csv", EXPORT)
raw_sentences = loading.load(DATA, "all_sentences.csv", EXPORT)
raw_metrics = loading.load(DATA, "all_metrics.csv", EXPORT)

loaded all_utterances.csv  1201 rows
loaded all_sentences.csv  9519 rows
loaded all_metrics.csv  510 rows


### Initial check — did the upload / extraction actually work?

Before doing any real analysis, we want a cheap sanity check: right shapes, right columns,
nothing obviously empty or mistyped. This is not EDA yet, it is just "did the file we expected
land correctly."

In [ ]:
print("Shapes:")
print(f"  metrics:    {raw_metrics.shape}")
print(f"  sentences:  {raw_sentences.shape}")
print(f"  utterances: {raw_utterances.shape}")

print("\nColumns:")
print(f"  metrics:    {list(raw_metrics.columns)}")
print(f"  sentences:  {list(raw_sentences.columns)}")
print(f"  utterances: {list(raw_utterances.columns)}")

print("\nFirst couple of rows of each, just to eyeball that parsing looks sane:")
print("\n--------------------------------- metrics.head() --------------------------")
print(raw_metrics.head(2))
print("\n----------------------------------- sentences.head() -----------------------------------")
print(raw_sentences.head(2))
print("\n----------------------------------- utterances.head() -----------------------------------")
print(raw_utterances.head(2))

In [ ]:
#----------------------------------------------------------------------------------------
# dtypes + missing-value counts: a quick structural check before we trust the data enough
# to explore it properly.
#----------------------------------------------------------------------------------------
for name, df in zip(["Metrics", "Sentences", "Utterances"], [raw_metrics, raw_sentences, raw_utterances]):
    print(f"\n=== {name} ===")
    print(df.dtypes)
    print(f"\nMissing values in {name}:")
    print(df.isnull().sum())

## Understand (diagnostic pass, on the raw data)

### What would we want to know about the data?

Once transcripts have been collected, our first job should be ordinary exploratory data analysis.

For example:

**Coverage**
- Which banks do we have?
- Which quarters do we have?
- Are there gaps?

**Document structure**
- Can we distinguish prepared remarks from Q&A?
- Can we identify individual speakers?
- Can we distinguish analysts from management?

**Volume**
- How long are transcripts?
- How many questions are asked each quarter?
- Does transcript length vary substantially between banks?

**Text quality**
- Are speaker labels reliable?
- Are tables or page headers mixed into the extracted text?
- Are there duplicated passages?
- Are there unusual characters or formatting problems?

These checks should come before topic modelling or sentiment analysis. Run at the raw-data
stage, they are also what tell us *what the Stage 2 structuring pipeline needs to fix* — three
findings below (an extra non-earnings call, a duplicated-speaker edge case, and a text-mangling
bug found later during a reconciliation check) directly shape the functions built further down.

### Coverage: which banks, which quarters, are there gaps?

In [ ]:
print("Banks present:", sorted(raw_sentences.bank.unique()))
print("Quarters present:", sorted(raw_sentences.quarter.unique()))

print("\nRow counts by bank x quarter (sentences):")
print(raw_sentences.groupby(["bank", "quarter"]).size())

print("\nDistinct call_date values per bank/quarter — should be exactly 1 unless a bank held")
print("an extra, non-routine call that quarter:")
print(raw_sentences.groupby(["bank", "quarter"]).call_date.nunique())
print(raw_sentences.groupby(["bank", "quarter"]).call_type.unique())

**Finding:** JPM 2023-Q2 has *two* calls, not one, the extra `call_type == "event"` row is
the First Republic acquisition call, not a routine quarterly earnings call. It is genuine
coverage, but pooling it with the routine earnings calls would distort any quarter-over-quarter
comparison later. We look at it directly below, then tag it explicitly in the Stage 2 pipeline
rather than dropping it.

In [ ]:
ev = raw_sentences[raw_sentences.call_type == "event"]
print("Sections present in the event call:", ev.section.unique())
print("Speaker roles present in the event call:", ev.speaker_role.unique())
print("Call date(s):", ev.call_date.unique())
print("\nSample sentences:")
print(ev.sentence.head(5).to_list())

### Document structure: prepared remarks vs. Q&A, individual speakers, analysts vs. management

In [ ]:
print("call_type values:", raw_sentences.call_type.unique())
print("section values:", raw_sentences.section.unique())
print("speaker_role values:", raw_sentences.speaker_role.unique())

print("\nsection x speaker_role crosstab (sentences):")
print(pd.crosstab(raw_sentences.section, raw_sentences.speaker_role))

print("\nUnique speaker_name count by role:")
print(raw_sentences.groupby("speaker_role").speaker_name.nunique())

print("\nAnalyst speaker_institution fill rate (do we know which sell-side firm each analyst is from?):")
an = raw_sentences[raw_sentences.speaker_role == "analyst"]
print(f"  {an.speaker_institution.isnull().mean():.1%} null; {an.speaker_institution.nunique()} unique institutions")
print(an.speaker_institution.dropna().unique()[:20])

**Finding:** `section` cleanly distinguishes prepared remarks from Q&A, and `speaker_role`
cleanly distinguishes analyst from management, so both of those checklist items are answered by
columns we already have. What is *not* directly usable is a stable per-call
"utterance id": the raw data only has `position_in_call`. We will need to deal with this later on in the process.

There is also a subtler issue worth flagging that we've come across, because it only shows up once we try to group
Q&A turns into exchanges: two different analysts can speak back-to-back with no management turn
in between (e.g. one analyst's closing "Thank you." immediately followed by a different analyst's
real question). A naive rule of "new exchange whenever `speaker_role` becomes `analyst`" would
silently merge those two people's questions into one exchange. We build the fix for this later on in the process, 
provisionally to be called: `build_qa_exchanges` (TBC)

### Volume: transcript length & how it varies

In [ ]:
print("--- Metrics: numerical summary ---")
print(raw_metrics.describe())

In [ ]:
#----------------------------------------------------------------------
# Sentence-level length metrics (words / characters per sentence)
#----------------------------------------------------------------------
raw_sentences["word_count"] = raw_sentences["sentence"].apply(lambda x: len(str(x).split()))
raw_sentences["char_count"] = raw_sentences["sentence"].apply(lambda x: len(str(x)))
print("--- Sentence text metrics ---")
print(raw_sentences[["word_count", "char_count"]].describe())

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(raw_sentences["word_count"], bins=15, kde=True, ax=axes[0], color="seagreen")
axes[0].set_title("Sentence Word Count Distribution", fontweight="bold")
axes[0].set_xlabel("Words per sentence")

sns.histplot(raw_sentences["char_count"], bins=15, kde=True, ax=axes[1], color="orange")
axes[1].set_title("Sentence Character Count Distribution", fontweight="bold")
axes[1].set_xlabel("Characters per sentence")

plt.tight_layout()
plt.show()

In [ ]:
#-------------------------------------------------------------------------
# Utterance-level length metrics (words / characters per speaker turn)
#-------------------------------------------------------------------------
raw_utterances["word_count"] = raw_utterances["text"].apply(lambda x: len(str(x).split()))
raw_utterances["char_count"] = raw_utterances["text"].apply(lambda x: len(str(x)))
print("--- Utterance text metrics ---")
print(raw_utterances[["word_count", "char_count"]].describe())
print("\nUtterance length by section (prepared remarks vs. Q&A tend to differ a lot):")
print(raw_utterances.groupby("section").text_length.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(raw_utterances["word_count"], bins=15, kde=True, ax=axes[0], color="seagreen")
axes[0].set_title("Utterance Word Count Distribution", fontweight="bold")
axes[0].set_xlabel("Words per utterance")

sns.histplot(raw_utterances["char_count"], bins=15, kde=True, ax=axes[1], color="orange")
axes[1].set_title("Utterance Character Count Distribution", fontweight="bold")
axes[1].set_xlabel("Characters per utterance")

plt.tight_layout()
plt.show()

**Finding:** utterances are noticeably longer than individual sentences (as expected, an
utterance is made of several sentences), and Q&A-section utterances behave differently from
prepared-remarks ones. The real question of "does length vary between banks" is better asked on
the *structured*, deduplicated data further down, once we get an handle on : `is_routine_earnings` which we can the use to compare like
with like.

### Text quality: speaker labels, duplicated passages, formatting artefacts

In [ ]:
#------------------------------------------------------------------------------------------------------
# Exact-duplicate sentence rows within the same call: a cheap first pass at "does the same
# passage appear twice", which can happen if page headers/footers or repeated boilerplate slipped
# into the extraction.
#--------------------------------------------------------------------------------------------------------
dup_rate = raw_sentences.duplicated(subset=["bank", "quarter", "call_date", "sentence"]).mean()
print(f"Exact duplicate sentence rows within the same call: {dup_rate:.2%}")
print("(non-zero doesn't necessarily mean a bug -- short filler lines like 'Thank you.' genuinely")
print(" repeat -- but a high rate is worth a manual look for header/footer contamination.)")

Above check only catches whole sentences repeating, this is not enough as it will miss a bug that mangles text within a sentence without duplicating anything. Conseqeuntly, we need a second, independent extraction of the same speech to check the first one agianst. For this, we can utilise the sentence-level file and the utterance-level file which were both extracted from the same transcript, so if both are correct, reconstructing an utterance from its component sentences should reproduce the utterance text exactly.

To accomplish this, first we need a way to say "these two rows are from the same call", we introcude the key at this point and continue to resue it, unchanged for the rest of this notebook including the proposed Stage 2 pipeline that  follows later.

In [28]:
#-----------------------------------------------------------------------------------
# Columns that uniquely identify one earnings/event call. Every join or
# groupby that needs to stay "within a single call" uses this same key, so
# rows from two different calls (or two calls in the same quarter, like the
# JPM 2023-Q2 event call) never get merged into one group by accident.
#------------------------------------------------------------------------------------
CALL_KEY = ["bank", "quarter", "call_type", "call_date"]

### Reconstruction check: does the sentence file agree with the utterance file?

Reconstruct each utterance's text by concatenating its sentences (in `sentence_number` order), line it up against that same
utterance's `text` field from the utterance-level file, and see how often and how they
disagree.

In [ ]:
def reconstruct_utterance_text(raw_sentences: pd.DataFrame) -> pd.DataFrame:
    """
    Rebuild each utterance's text by concatenating its sentences, in order. Returns one row
    per (CALL_KEY, position_in_call), the same granularity as one row of raw_utterances
    so the two can be compared directly.
    """
    ordered = raw_sentences.sort_values(CALL_KEY + ["position_in_call", "sentence_number"])
    return (
        ordered.groupby(CALL_KEY + ["position_in_call"])["sentence"]
        .apply(lambda s: " ".join(s))
        .rename("reconstructed_text")
        .reset_index()
    )


def normalise_whitespace(text: str) -> str:
    """
    Collapse any run of whitespace to a single space, so a join-related spacing
    difference doesn't get mistaken for a real text disagreement.
    """
    return re.sub(r"\s+", " ", str(text)).strip()


reconstructed = reconstruct_utterance_text(raw_sentences)
recon_check = raw_utterances.merge(reconstructed, on=CALL_KEY + ["position_in_call"], how="inner")

recon_check["text_norm"] = recon_check["text"].apply(normalise_whitespace)
recon_check["reconstructed_norm"] = recon_check["reconstructed_text"].apply(normalise_whitespace)
recon_check["matches_exactly"] = recon_check["text_norm"] == recon_check["reconstructed_norm"]

print(f"Utterances checked: {len(recon_check)}")
print(f"Exact character-for-character match between the two extractions: {recon_check['matches_exactly'].mean():.1%}")

**Findings:** The match rate is 78.5% indicating the two files disagree somehow or somewhere but not exactly sure where or by what. From granulaity perspective, it makes sense to compare (diff) at the word level instead of characters, this way it shoud be possible to see where teh differences occurs or what is responsible for it.

In [ ]:
import difflib
from collections import Counter


def word_level_diffs(text_a: str, text_b: str) -> list:
    """
    Where two texts disagree, return the (words_from_a, words_from_b) chunks that differ.
   
    """
    words_a, words_b = text_a.split(), text_b.split()
    matcher = difflib.SequenceMatcher(a=words_a, b=words_b)
    diffs = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            diffs.append((tuple(words_a[i1:i2]), tuple(words_b[j1:j2])))
    return diffs


mismatches = recon_check[~recon_check["matches_exactly"]]
print(f"{len(mismatches)} utterances disagree between the two extractions.\n")

all_diffs = []
for _, row in mismatches.iterrows():
    all_diffs.extend(word_level_diffs(row["text_norm"], row["reconstructed_norm"]))

print("Most common word-level disagreements (utterance-file wording -> sentence-file wording):")
for (from_utterances, from_sentences), n in Counter(all_diffs).most_common(15):
    print(f"  {n:>4}x   {' '.join(from_utterances)!r:>20}  ->  {' '.join(from_sentences)!r}")

**Findings:**  The outcome of application of the diff process shows that the process which extracted the raw sentences from the utterances indicated that some words were not properly transformed on the sentences side. The word `do not` on the uttereance files shows up as `donot` on the sentence file. As an example; the following also follows the same patterns for: `does not` becomes `doesnot`, `was not` becomes `wasnot` , e.t.c. 

In summary, above indicated that `all_sentences` misconstrcuted  the contractions during the process whoch produced it; the apostrphe `'` and `t'` was expnded to `not` but the required space was dropped. Th ereason we didn't see `cannot` is because it is already teh correct expansion of `can't` and therefore does not require fixing.

In an answer to on eof the rubrics / checklist: `are speaker labels reliable / is there a formatting problem` question, the answer from the findings is that it isn't.  Moving forward, we will now fix this contractions problem in the pipeline which builds the structured utterance and sentence but only need to do it in in one place since it only happened at sentences-level.



### Summary of EDA findings

All of our findings showed:
- the raw tables are coverage-complete
- the raw tables are usably structured

However, to make it trustworthy  as an inputs to semnantic/topic stages, we need to apply a thin layer of cleanup:
- stable ID scheme
- one text fix
- one grouping fix




## Structure

Turn the raw extracted tables into a single, consistent, well-keyed dataset that later stagesand other team members can rely on without having to derive structured textx individually.



### Utility  / helper  functions

We implements some utility / helper functions where each function takes as input a DataFrame and returns a new one (changes are never made to the input) 
The `helper / utility` functions is implemented in `src` directory and will be imported for use in the pipeline implementaion.


## Orchestration: Build structure pipeine

In [31]:
from src import utils



In [ ]:
data = utils.run_pipeline()

print("Row counts:")
for name in ["raw_sentences", "raw_utterances", "raw_metrics",
             "structured_utterances", "structured_sentences", "qa_exchanges"]:
    print(f"  {name:24s} {len(data[name])} rows")

# NB: is_filler is repeated on every TURN within an exchange, so we must
# de-duplicate down to one row per exchange before counting, summing
# the raw column would double-count any multi-turn exchange.
distinct_exchanges = data["qa_exchanges"].drop_duplicates(utils.CALL_KEY + ["qa_exchange_id"])
print(f"\nQ&A exchanges: {len(distinct_exchanges)} total, "
      f"{distinct_exchanges['is_filler'].sum()} filler, "
      f"{(~distinct_exchanges['is_filler']).sum()} substantive")

# Persist structured outputs for downstream stages / sharing.
data["structured_utterances"].to_csv("structured_utterances.csv", index=False)
data["structured_sentences"].to_csv("structured_sentences.csv", index=False)
data["qa_exchanges"].to_csv("qa_exchanges_utterance_level.csv", index=False)
print("\nSaved structured_utterances.csv, structured_sentences.csv, qa_exchanges_utterance_level.csv")

In [ ]:
def exchanges_per_quarter(qa: pd.DataFrame, include_filler: bool = True) -> pd.DataFrame:
    """
    Count distinct Q&A exchanges per (bank, quarter, call_type).

    Takes a DataFrame in, returns a new one: `qa` itself (and therefore
    data["qa_exchanges"]) is never modified by calling this.

    include_filler=False restricts to the substantive exchanges used for
    topic modelling; True (default) counts every real question-driven
    exchange, matching the coverage view used in the diagnostic EDA above.
    """
    df = qa if include_filler else qa[~qa["is_filler"]]
    return (
        df.drop_duplicates(subset=CALL_KEY + ["qa_exchange_id"])
        .groupby(["bank", "quarter", "call_type"])
        .size()
        .rename("n_exchanges")
        .reset_index()
    )


counts = exchanges_per_quarter(data["qa_exchanges"], include_filler=True)
counts.sort_values(["bank", "quarter"])

### Q&A exchanges per quarterly call



In [ ]:
def plot_exchanges_per_quarter(counts: pd.DataFrame):
    """
    Builds the figure and returns it.
    `fig.savefig(...)` is available if we want to save it as a PNG
    for a slide or report.
    """
    quarters = sorted(counts["quarter"].unique())
    fig, ax = plt.subplots(figsize=(9, 5))

    for bank, style in [("JPM", dict(color="#0f4c81", marker="o")),
                         ("UBS", dict(color="#d64541", marker="s"))]:
        routine = (counts[(counts.bank == bank) & (counts.call_type == "earnings")]
                   .set_index("quarter").reindex(quarters))
        ax.plot(quarters, routine["n_exchanges"], label=bank, linewidth=2, **style)

        special = counts[(counts.bank == bank) & (counts.call_type != "earnings")]
        if not special.empty:
            ax.scatter(special["quarter"], special["n_exchanges"],
                       s=140, facecolors="none", edgecolors=style["color"],
                       linewidths=2, zorder=5)
            for _, row in special.iterrows():
                ax.annotate(f"{row.call_type} call", (row["quarter"], row["n_exchanges"]),
                            textcoords="offset points", xytext=(0, 10),
                            fontsize=8, color=style["color"])

    ax.set_title("Q&A exchanges per quarterly call")
    ax.set_ylabel("exchanges")
    ax.set_xticks(range(len(quarters)))
    ax.set_xticklabels(quarters, rotation=0)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    return fig


fig = plot_exchanges_per_quarter(counts)
# Uncomment to save a copy for a report/slide:
# fig.savefig("qa_exchanges_per_quarter.png", dpi=150)

Routines earnings calls are displayed as a continou sline per bank; the JPM 2023-Q2 event call (First Republic acquistion) is plotted in its own right separted from the line since it's not a routine quarterly call and would distort the quatrrt-on-quater trend.

### Hands-off summary

This concludes **Stage 3** of the project:
The clean documented inputs required by the later stages are persisted as CSVs:
-  `data["structured_sentences"]`,
-  `data["structured_utterances"]` 
-  `data["qa_exchanges"]` 

Anyone continuing this work should be able to start from those three files, plus this notebook as
the record of *why* they look the way they do without re-deriving any of the fixes above. 